# Storm and Zillow combined data

$Price_{T+1} = a + B*Range + C*TotalDamage + D*Duration + E*NationalChange$

## Data Loading helper classes

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pytest
import seaborn as sns
import datetime
import numpy as np
import pyfixest as pf

In [2]:
noaa_2025_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260218.csv.gz"
noaa_2024_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260116.csv.gz"
noaa_2023_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2023_c20260116.csv.gz"
noaa_2022_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2022_c20250721.csv.gz"
noaa_2021_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2021_c20250520.csv.gz"
noaa_2020_details = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2020_c20260116.csv.gz"


noaa_2025_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2025_c20260218.csv"
noaa_2024_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2024_c20260116.csv.gz"
noaa_2023_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2023_c20260116.csv.gz"
noaa_2022_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2022_c20250721.csv.gz"
noaa_2021_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2021_c20250520.csv.gz"
noaa_2020_locations = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_locations-ftp_v1.0_d2020_c20260116.csv.gz"

zillow_county_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1769612738"
zillow_msa_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1769612738"

In [3]:
class NOAAStormParser:
    def __init__(self, details_url, locations_url):
        details_df = pd.read_csv(details_url)
        locations_df = pd.read_csv(locations_url)
        self.df = pd.merge(locations_df, details_df, on="EVENT_ID", how="left")
        self.clean()
        
    def clean(self):
        self.parse_dates()
        self.parse_damages()
        self.parse_storm_duration()

    def parse_dates(self):
        self.df["YEARMONTH"] = pd.to_datetime(self.df["YEARMONTH"], format='%Y%m') + pd.offsets.MonthEnd(0)
        
    def parse_storm_duration(self):
        self.df["BEGIN_DATE_TIME"] = pd.to_datetime(self.df["BEGIN_DATE_TIME"])
        self.df["END_DATE_TIME"] = pd.to_datetime(self.df["END_DATE_TIME"])
        duration_delta = self.df["END_DATE_TIME"] - self.df["BEGIN_DATE_TIME"]
        self.df["DURATION_HOURS"] = duration_delta.dt.total_seconds() / 3600.0
        
    def parse_damages(self):
        self.df['DAMAGE_PROPERTY'] = self.df["DAMAGE_PROPERTY"].apply(self.parse_numeric_string)
        self.df['DAMAGE_CROPS'] = self.df["DAMAGE_CROPS"].apply(self.parse_numeric_string)
        self.df['TOTAL_DAMAGE'] = self.df["DAMAGE_CROPS"] + self.df["DAMAGE_PROPERTY"]

    def parse_numeric_string(self, rawstring):
        if pd.isna(rawstring):
            return 0

        rawstring = str(rawstring) 
        
        if "K" in rawstring:
            value = rawstring.replace("K","")
            return float(value) * 1000
        if "M" in rawstring:
            value = rawstring.replace("M","")
            return float(value) * 1_000_000
        if "B" in rawstring:
            value = rawstring.replace("B","")
            return float(value) * 1_000_000_000
        return float(rawstring)


In [4]:
class ZillowDataParser():
    DTYPES = {
        'RegionID': 'Int64',
        'SizeRank': 'Int64',
        'RegionName': 'object',
        'RegionType': 'object',
        'StateName': 'object',
        'State': 'object',
        'Metro': 'object',
        'StateCodeFIPS': 'Int64',
        'MunicipalCodeFIPS': 'Int64'
    }
    
    META_DATA_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
                      'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
                      'latitude', 'longitude']
    
    def __init__(self, regional_url, national_url):
        self.download_data(regional_url, national_url)
        
    def download_data(self, regional_url, national_url):
        regional_data_raw = pd.read_csv(regional_url)
        national_data_raw = pd.read_csv(national_url)
        national = national_data_raw.loc[national_data_raw["RegionName"] == "United States"]
        self.df = pd.concat([regional_data_raw, national],sort=False, ignore_index=True)
        self.df = self.df.astype(self.DTYPES)
        all_columns = self.df.columns.tolist() + [col for col in self.META_DATA_COLS if col not in self.df.columns]
        self.df = self.df.reindex(columns=all_columns)
        
    def fips_time_series(self, fips: tuple = ()):
        """
        Returns DataFrame with one column of time series data
        """
        regional = self.df.loc[(self.df['StateCodeFIPS'] == fips[0]) & (self.df['MunicipalCodeFIPS'] == fips[1])]
        
        if regional.empty:
            return pd.DataFrame()
        
        regional_data = regional.loc[:, self.date_cols()].T
        
        region_name = regional['RegionName'].iloc[0]
        new_columns = pd.MultiIndex.from_tuples(
            [(region_name, fips[0], fips[1])],
            names=['Region', 'StateFIPS', 'CountyFIPS']
        )
        
        regional_data.columns = new_columns
        regional_data.index = pd.to_datetime(regional_data.index)
        return regional_data
    
    def normalized_from_date(self, start_date, months_back=0):
        """
        Returns a DataFrame that is scaled to 100 as of the start_date
        """
        start_date = pd.to_datetime(start_date)
        
        
        if months_back > 0:
            window_start_date = start_date - pd.DateOffset(months=months_back)
        else:
            window_start_date = start_date
        
        all_dates = self.date_cols()
        result_dates = [date for date in all_dates if pd.to_datetime(date) >= window_start_date]
        
        if not result_dates:
            return pd.DataFrame()
        ref_col = min(result_dates, key=lambda x: abs(pd.to_datetime(x) - start_date))
        
        result_cols = self.META_DATA_COLS + result_dates        
        normalized_df = self.df[result_cols].copy()
        baseline = normalized_df[ref_col] / 100.0
        normalized_df[result_dates] = normalized_df[result_dates].div(baseline, axis=0)
        return normalized_df

    def change_after_event(self, event_date, fips):
        """
        Returns a DataFrame with the normalized index for a set of region
        1, 2, 3, 6, 12 months after an event data
        """
        event_date = datetime(event_date)
        norm_df = self.normalized_from_date(event_date, months_back=24)
        country_mask = norm_df["RegionType"] == "country"
        fips_mask = (norm_df['StateCodeFIPS'] == fips[0]) & (norm_df['MunicipalCodeFIPS'] == fips[1])
        norm_df = norm_df[fips_mask | country_mask]
        available_dates = [col for col in norm_df.columns if col not in self.META_DATA_COLS]
        event_date = pd.to_datetime(event_date)
        intervals = list(range(1,13))
        results = norm_df[self.META_DATA_COLS].copy()
        
        for m in intervals:
            plus_date = event_date + pd.DateOffset(months=m)
            closest_date = min(available_dates, key=lambda x: abs(pd.to_datetime(x) - plus_date))
            
            if abs(pd.to_datetime(closest_date) - plus_date).days < 15:
                 results[f'plus_{m}_mon'] = norm_df[closest_date]
            else:
                 results[f'plus_{m}_mon'] = np.nan
            
            minus_date = event_date - pd.DateOffset(months=m)
            closest_minus = min(available_dates, key=lambda x: abs(pd.to_datetime(x) - minus_date))
            if abs(pd.to_datetime(closest_minus) - minus_date).days < 15:
                results[f'minus_{m}_mon'] = norm_df[closest_minus]
            else:
                results[f'minus_{m}_mon'] = np.nan
        
        results.reset_index(inplace=True, drop=True)
        return results

    def latest_date(self):
        return self.date_cols()[-1]

    def date_cols(self):
        dates = [col for col in self.df.columns if col not in self.META_DATA_COLS]
        dates = sorted(dates, key=pd.to_datetime)
        return list(dates)
    
    def get_monthly_panel(self):
        """
        Transforms the Wide Zillow data into a Long format panel.
        """
        date_cols = [col for col in self.df.columns if col not in self.META_DATA_COLS]

        long_df = self.df.melt(
            id_vars=self.META_DATA_COLS, 
            value_vars=date_cols,
            var_name='Date', 
            value_name='Index_Value'
        )
        

        long_df['Date'] = pd.to_datetime(long_df['Date'])
        long_df = long_df.sort_values(['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'])
        long_df['Regional_Pct_Change'] = long_df.groupby(['StateCodeFIPS', 'MunicipalCodeFIPS'])['Index_Value'].pct_change()

        national_df = long_df[long_df['RegionName'] == 'United States'].copy()
        national_df = national_df.sort_values('Date')
        national_df['Regional_Pct_Change'] = national_df['Index_Value'].pct_change()

        national_df = national_df[['Date', 'Index_Value', 'Regional_Pct_Change']]
        national_df = national_df.rename(columns={
            'Index_Value': 'National_Index',
            'Regional_Pct_Change': 'National_Pct_Change'
        })

        final_df = pd.merge(long_df, national_df, on='Date', how='left')

        final_df = final_df.set_index(['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'])
        
        return final_df

## Load  Data

In [5]:
noaa2025 = NOAAStormParser(noaa_2025_details, noaa_2025_locations) 
noaa2024 = NOAAStormParser(noaa_2024_details, noaa_2024_locations) 
noaa2023 = NOAAStormParser(noaa_2023_details, noaa_2023_locations) 
noaa2022 = NOAAStormParser(noaa_2022_details, noaa_2022_locations) 
noaa2021 = NOAAStormParser(noaa_2021_details, noaa_2021_locations) 
noaa2020 = NOAAStormParser(noaa_2020_details, noaa_2020_locations) 

/var/folders/pz/bg9vnbfd2lx6cs0yw9q7vc3w0000gn/T/ipykernel_65501/2744332134.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.df["BEGIN_DATE_TIME"] = pd.to_datetime(self.df["BEGIN_DATE_TIME"])
/var/folders/pz/bg9vnbfd2lx6cs0yw9q7vc3w0000gn/T/ipykernel_65501/2744332134.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.df["END_DATE_TIME"] = pd.to_datetime(self.df["END_DATE_TIME"])
/var/folders/pz/bg9vnbfd2lx6cs0yw9q7vc3w0000gn/T/ipykernel_65501/2744332134.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.df["BEGIN_DATE_TIME"] = pd.to_datetime(self.df["

In [21]:
zillow = ZillowDataParser(zillow_county_url, zillow_msa_url)

In [36]:
zillow.get_monthly_panel()

/var/folders/pz/bg9vnbfd2lx6cs0yw9q7vc3w0000gn/T/ipykernel_65501/1976653270.py:134: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  long_df['Regional_Pct_Change'] = long_df.groupby(['StateCodeFIPS', 'MunicipalCodeFIPS'])['Index_Value'].pct_change()


RegionID  SizeRank  \
StateCodeFIPS MunicipalCodeFIPS Date                             
1             1                 2000-01-31      1524       904   
                                2000-02-29      1524       904   
                                2000-03-31      1524       904   
                                2000-04-30      1524       904   
                                2000-05-31      1524       904   
...                                              ...       ...   
<NA>          <NA>              2025-09-30    102001         0   
                                2025-10-31    102001         0   
                                2025-11-30    102001         0   
                                2025-12-31    102001         0   
                                2026-01-31    102001         0   

                                                RegionName RegionType  \
StateCodeFIPS MunicipalCodeFIPS Date                                    
1             1                 2000-01-31  Autauga County     county   
                                2000-02-29  Autauga County     county   
                                2000-03-31  Autauga County     county   
                                2000-04-30  Autauga County     county   
                                2000-05-31  Autauga County     county   
...                                                    ...        ...   
<NA>          <NA>              2025-09-30   United States    country   
                                2025-10-31   United States    country   
                                2025-11-30   United States    country   
                                2025-12-31   United States    country   
                                2026-01-31   United States    country   

                                           StateName State           Metro  \
StateCodeFIPS MunicipalCodeFIPS Date                                         
1             1                 2000-01-31        AL    AL  Montgomery, AL   
                                2000-02-29        AL    AL  Montgomery, AL   
                                2000-03-31        AL    AL  Montgomery, AL   
                                2000-04-30        AL    AL  Montgomery, AL   
                                2000-05-31        AL    AL  Montgomery, AL   
...                                              ...   ...             ...   
<NA>          <NA>              2025-09-30       NaN   NaN             NaN   
                                2025-10-31       NaN   NaN             NaN   
                                2025-11-30       NaN   NaN             NaN   
                                2025-12-31       NaN   NaN             NaN   
                                2026-01-31       NaN   NaN             NaN   

                                            latitude  longitude  \
StateCodeFIPS MunicipalCodeFIPS Date                              
1             1                 2000-01-31       NaN        NaN   
                                2000-02-29       NaN        NaN   
                                2000-03-31       NaN        NaN   
                                2000-04-30       NaN        NaN   
                                2000-05-31       NaN        NaN   
...                                              ...        ...   
<NA>          <NA>              2025-09-30       NaN        NaN   
                                2025-10-31       NaN        NaN   
                                2025-11-30       NaN        NaN   
                                2025-12-31       NaN        NaN   
                                2026-01-31       NaN        NaN   

                                              Index_Value  \
StateCodeFIPS MunicipalCodeFIPS Date                        
1             1                 2000-01-31  119748.046840   
                                2000-02-29  119777.972655   
                                2000-03-31  119581.507065   
                                2000-04-30 

## Combine Data
- Each location, pull the event data from Zillow for regional and national changes

In [23]:
combined_noaa_df = pd.concat([noaa2025.df, noaa2024.df, noaa2023.df, noaa2023.df, noaa2021.df, noaa2020.df], ignore_index=True)

In [24]:
combined_noaa_df.columns

Index(['YEARMONTH', 'EPISODE_ID_x', 'EVENT_ID', 'LOCATION_INDEX', 'RANGE',
       'AZIMUTH', 'LOCATION', 'LATITUDE', 'LONGITUDE', 'LAT2', 'LON2',
       'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH',
       'END_DAY', 'END_TIME', 'EPISODE_ID_y', 'STATE', 'STATE_FIPS', 'YEAR',
       'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO',
       'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT',
       'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT',
       'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE',
       'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE',
       'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE',
       'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE',
       'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH',
       'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
       'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'TOTAL_DAMAGE',
   

In [ ]:
# Aggregate the weather events by month and location
agg_rules = {
    'EVENT_ID': 'count',
    'DURATION_HOURS': 'sum',
    'TOTAL_DAMAGE': 'sum',
    
}
monthly_summary = combined_noaa_df.groupby(['STATE_FIPS', 'CZ_FIPS', 'YEARMONTH']).agg(agg_rules).reset_index()

# Convert duration to days
monthly_summary['DURATION_HOURS'] = monthly_summary['DURATION_HOURS'] / 24

# Convert damage to billions USD
monthly_summary['TOTAL_DAMAGE'] = monthly_summary['TOTAL_DAMAGE'] / 1_000_000_000

monthly_summary = monthly_summary.rename(columns={
    'EVENT_ID': 'EVENT_COUNT',
    'DURATION_HOURS': 'TOTAL_DURATION_DAYS',
    'TOTAL_DAMAGE': 'TOTAL_DAMAGE_BILLIONS'
})

In [26]:
def merge_with_lag(housing_df, storm_df, lag_months=1):
    lagged_storms = storm_df.copy()

    rename_map = {
        'STATE_FIPS': 'StateCodeFIPS',
        'CZ_FIPS': 'MunicipalCodeFIPS',
        'YEARMONTH': 'Date'
    }
    lagged_storms = lagged_storms.rename(columns=rename_map)

    if not pd.api.types.is_datetime64_any_dtype(lagged_storms['Date']):
        lagged_storms['Date'] = pd.to_datetime(lagged_storms['Date'])

    lagged_storms['Date'] = lagged_storms['Date'] + pd.DateOffset(months=lag_months)

    if 'Date' not in housing_df.columns:
        housing_temp = housing_df.reset_index()
    else:
        housing_temp = housing_df.copy()

    merged_df = pd.merge(
        housing_temp,
        lagged_storms,
        on=['StateCodeFIPS', 'MunicipalCodeFIPS', 'Date'],
        how='left' # left to keep all the non event data
    )
    storm_cols = ['EVENT_COUNT', 'TOTAL_DURATION_DAYS', 'TOTAL_DAMAGE_BILLIONS']
    merged_df[storm_cols] = merged_df[storm_cols].fillna(0)
    return merged_df

In [110]:
lag_months=3
df = merge_with_lag(zillow.get_monthly_panel(), monthly_summary, lag_months)

/var/folders/pz/bg9vnbfd2lx6cs0yw9q7vc3w0000gn/T/ipykernel_65501/1976653270.py:134: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  long_df['Regional_Pct_Change'] = long_df.groupby(['StateCodeFIPS', 'MunicipalCodeFIPS'])['Index_Value'].pct_change()


### FEMA NRI

In [111]:
df = df.drop(columns=['latitude', 'longitude'])

# build 5-digit county FIPS
df['county_fips'] = (
    df['StateCodeFIPS'].astype(str).str.zfill(2) +
    df['MunicipalCodeFIPS'].astype(str).str.zfill(3)
)

# build year-month and state-month fixed effect column
df['year_month'] = df['Date'].dt.to_period('M').astype(str)  # e.g. '2018-03'
df['state_month'] = df['StateCodeFIPS'].astype(str).str.zfill(2) + '_' + df['year_month']

# log transform the dependent variable
df['log_index'] = np.log(df['Index_Value'])

In [112]:
nri = pd.read_csv('../data/NRI_Long.csv')
nri['fips'] = nri['nri_id'].str.replace("C","")
nri = nri.drop(["Unnamed: 0"], axis=1)
nri.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12579 entries, 0 to 12578
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nri_id      12579 non-null  object 
 1   stateabbrv  12579 non-null  object 
 2   county      12579 non-null  object 
 3   risk_value  12579 non-null  float64
 4   eal_valt    12579 non-null  float64
 5   sovi_score  12579 non-null  float64
 6   resl_score  12579 non-null  float64
 7   crf_value   12579 non-null  float64
 8   year        12579 non-null  int64  
 9   fips        12579 non-null  object 
dtypes: float64(5), int64(1), object(4)
memory usage: 982.9+ KB


In [113]:
def assign_nri_vintage(date):
    year = date.year
    if year < 2021:
        return 2020
    elif year < 2023:
        return 2021
    elif year < 2025:
        return 2023
    else:
        return 2025

df['nri_vintage'] = df['Date'].apply(assign_nri_vintage)

# merge on both fips and vintage year
df = df.merge(
    nri[['fips', 'year', 'risk_value', 'eal_valt', 
                  'sovi_score', 'resl_score', 'crf_value']],
    left_on=['county_fips', 'nri_vintage'],
    right_on=['fips', 'year'],
    how='left'
)

df = df.drop(columns=['fips', 'year'])

#### Incorporating fixed effects

In [114]:
df = df.dropna(subset=['resl_score', 'risk_value'])
df = df.dropna(subset=['StateCodeFIPS', 'MunicipalCodeFIPS', 'Index_Value'])


# build fixed effect columns
df['county_fips'] = (df['StateCodeFIPS'].astype(str).str.zfill(2) + 
                     df['MunicipalCodeFIPS'].astype(str).str.zfill(3))
df['year_month'] = df['Date'].dt.to_period('M').astype(str)
df['state_month'] = df['StateCodeFIPS'].astype(str).str.zfill(2) + '_' + df['year_month']
df['log_index'] = np.log(df['Index_Value'])
df['log_damage'] = np.log1p(df['TOTAL_DAMAGE_BILLIONS'])
df['resl_score_std'] = (df['resl_score'] - df['resl_score'].mean()) / df['resl_score'].std()
df['log_national'] = np.log(df['National_Index'])

df = df[df['RegionType']=='county']

In [118]:
model = pf.feols(
    "log_index ~ log_damage + log_damage:resl_score_std | county_fips + state_month",
    data=df,
    vcov={"CRV1": "county_fips"}
)
print(f"Months Lag = {lag_months}")
model.summary()

/opt/anaconda3/envs/mids_env/lib/python3.11/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 881 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(


Months Lag = 3
###

Estimation:  OLS
Dep. var.: log_index, Fixed effects: county_fips+state_month
Inference:  CRV1
Observations:  684398

| Coefficient               |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| log_damage                |      0.030 |        0.023 |     1.326 |      0.185 | -0.015 |   0.076 |
| log_damage:resl_score_std |     -0.041 |        0.022 |    -1.871 |      0.061 | -0.084 |   0.002 |
---
RMSE: 0.077 R2: 0.978 R2 Within: 0.0 


In [ ]:
model = pf.feols(
    "log_index ~ log_damage + log_damage:resl_score_std + log_national",
    data=df,
    vcov={"CRV1": "county_fips"}
)
print(f"Months Lag = {lag_months}")
model.summary()

###

Estimation:  OLS
Dep. var.: log_index, Fixed effects: 0
Inference:  CRV1
Observations:  685279

| Coefficient               |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept                 |      3.004 |        0.117 |    25.745 |      0.000 |  2.775 |   3.233 |
| log_damage                |      0.559 |        0.116 |     4.809 |      0.000 |  0.331 |   0.787 |
| log_national              |      0.727 |        0.009 |    78.154 |      0.000 |  0.708 |   0.745 |
| log_damage:resl_score_std |     -0.005 |        0.088 |    -0.058 |      0.954 | -0.179 |   0.168 |
---
RMSE: 0.468 R2: 0.179 


In [120]:
model = pf.feols(
    "log_index ~ log_damage + log_damage:resl_score_std + log_national | county_fips + state_month",
    data=df,
    vcov={"CRV1": "county_fips"}
)
print(f"Months Lag = {lag_months}")
model.summary()

/opt/anaconda3/envs/mids_env/lib/python3.11/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 881 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(
/opt/anaconda3/envs/mids_env/lib/python3.11/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['log_national'].
            
  warnings.warn(


Months Lag = 3
###

Estimation:  OLS
Dep. var.: log_index, Fixed effects: county_fips+state_month
Inference:  CRV1
Observations:  684398

| Coefficient               |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| log_damage                |      0.030 |        0.023 |     1.326 |      0.185 | -0.015 |   0.076 |
| log_damage:resl_score_std |     -0.041 |        0.022 |    -1.871 |      0.061 | -0.084 |   0.002 |
---
RMSE: 0.077 R2: 0.978 R2 Within: 0.0 


In [121]:
df['log_event_count'] = np.log1p(df['EVENT_COUNT'])

model_count = pf.feols(
    "log_index ~ log_event_count + log_event_count:resl_score_std | county_fips + state_month",
    data=df,
    vcov={"CRV1": "county_fips"}
)
print(f"Months Lag = {lag_months}")
model_count.summary()

/opt/anaconda3/envs/mids_env/lib/python3.11/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 881 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(


Months Lag = 3
###

Estimation:  OLS
Dep. var.: log_index, Fixed effects: county_fips+state_month
Inference:  CRV1
Observations:  684398

| Coefficient                    |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| log_event_count                |      0.003 |        0.001 |     3.632 |      0.000 |  0.001 |   0.004 |
| log_event_count:resl_score_std |      0.001 |        0.000 |     2.514 |      0.012 |  0.000 |   0.002 |
---
RMSE: 0.077 R2: 0.978 R2 Within: 0.0 


### Storm Severity


how does the home value index change with teh severity of storm events?
Look at the log change in home value index regressed on the log of damage per event with county level fixed effects



In [122]:
df['damage_per_event'] = df['TOTAL_DAMAGE_BILLIONS'] / df['EVENT_COUNT'].replace(0, np.nan)
df['damage_per_event'] = df['damage_per_event'].fillna(0)
df['log_damage_per_event'] = np.log1p(df['damage_per_event'])

model_count = pf.feols(
    "log_index ~ log_damage_per_event + log_damage_per_event:resl_score_std | county_fips + state_month",
    data=df,
    vcov={"CRV1": "county_fips"}
)
print(f"Months Lag = {lag_months}")
model_count.summary()

/opt/anaconda3/envs/mids_env/lib/python3.11/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 881 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(


Months Lag = 3
###

Estimation:  OLS
Dep. var.: log_index, Fixed effects: county_fips+state_month
Inference:  CRV1
Observations:  684398

| Coefficient                         |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| log_damage_per_event                |      0.184 |        0.057 |     3.245 |      0.001 |  0.073 |   0.296 |
| log_damage_per_event:resl_score_std |     -0.796 |        0.232 |    -3.434 |      0.001 | -1.250 |  -0.341 |
---
RMSE: 0.077 R2: 0.978 R2 Within: 0.0 


In [123]:
df['resl_score_std'].describe() 

count    6.852790e+05
mean     1.266138e-15
std      1.000000e+00
min     -4.489078e+00
25%     -1.381579e-01
50%      5.807506e-02
75%      2.567171e-01
max      3.814946e+00
Name: resl_score_std, dtype: float64

In [125]:
df['storm_effect'] = 0.184 + (-0.796 * df['resl_score_std'])
print(f"Share with positive effect: {(df['storm_effect'] > 0).sum() / len(df)}")
print(f"Share with negative effect: {(df['storm_effect'] < 0).sum() / len(df)}")


Share with positive effect: 0.7239036946995311
Share with negative effect: 0.27609630530046886


The log damage per event is significant at the $p=0.001$ level with a 3 month delay. 3 months after storm events requires rebuilding so home prices increase by 0.184%.
Counties with higher resilience scores see a dampened price response. 